### GTP-generated, input: predicted dataset, output: pdf with correlation matrix and classification_report

In [ ]:
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import datetime as dt

now = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Setup file paths
results_dir = '../dataset/results'
pdf_output_path = os.path.join(results_dir, f'all_datasets_evaluation_report_{now}.pdf')

# Find prediction CSV files
pred_files = sorted(glob.glob(os.path.join(results_dir, 'pred_*_xgboost.csv')))

# Fallback search if filenames differ slightly
if not pred_files:
    pred_files = sorted([
        os.path.join(results_dir, f) for f in os.listdir(results_dir)
        if f.endswith('.csv') and 'evaluation' not in f and 'summary' not in f
    ])

# Open a multi-page PDF context
with PdfPages(pdf_output_path) as pdf:
    for file_path in pred_files:
        file_name = os.path.basename(file_path)
        df = pd.read_csv(file_path)

        y_true = df['Actual_Activity']
        y_pred = df['Predicted_Activity']

        labels = sorted(list(set(y_true).union(set(y_pred))))

        # 1. Compute Metrics
        cls_dict = classification_report(y_true, y_pred, output_dict=True)
        cm = confusion_matrix(y_true, y_pred, labels=labels)

        # ------------------------------------------------------------------
        # Normalize Matrix by Row (True Classes) into Percentages (0% - 100%)
        # ------------------------------------------------------------------
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

        # Setup Figure Layout (A4 size portrait: 8.5 x 11 inches)
        fig = plt.figure(figsize=(8.5, 11))
        
        # Title Block
        plt.suptitle(
            f"Dataset: {file_name}\n"
            f"Overall Accuracy: {cls_dict['accuracy']:.4f}  |  Macro F1-Score: {cls_dict['macro avg']['f1-score']:.4f}",
            fontsize=12,
            fontweight='bold',
            y=0.96
        )

        # Subplot 1: Seaborn Normalized Heatmap (Percentages)
        ax1 = fig.add_subplot(2, 1, 1)
        
        # Format string array with '%' signs inside the heatmap cells
        annot_labels = np.array([
            [f"{val:.1f}%" for val in row] for row in cm_normalized
        ])

        sns.heatmap(
            cm_normalized,
            annot=annot_labels,
            fmt="",  # Using custom string formatting with % signs
            cmap='Blues',
            xticklabels=labels,
            yticklabels=labels,
            cbar=True,
            cbar_kws={'label': 'Percentage (%)'},
            vmin=0,
            vmax=100,
            ax=ax1
        )
        
        ax1.set_title('Normalized Matrix (Actual vs. Predicted %)', fontsize=11, fontweight='bold', pad=10)
        ax1.set_xlabel('Predicted Activity', fontsize=9, fontweight='bold')
        ax1.set_ylabel('Actual Activity', fontsize=9, fontweight='bold')
        plt.setp(ax1.get_xticklabels(), rotation=30, ha='right', fontsize=8)
        plt.setp(ax1.get_yticklabels(), rotation=0, fontsize=8)

        # Subplot 2: Classification Report Table (Bottom Half)
        ax2 = fig.add_subplot(2, 1, 2)
        ax2.axis('off')

        table_data = []
        headers = ['Class / Metric', 'Precision', 'Recall', 'F1-Score', 'Support']

        for key, value in cls_dict.items():
            if isinstance(value, dict):
                table_data.append([
                    key,
                    f"{value['precision']:.4f}",
                    f"{value['recall']:.4f}",
                    f"{value['f1-score']:.4f}",
                    int(value['support'])
                ])
            elif key == 'accuracy':
                table_data.append(['accuracy', '', '', f"{value:.4f}", len(y_true)])

        table = ax2.table(
            cellText=table_data,
            colLabels=headers,
            loc='center',
            cellLoc='center'
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1.0, 1.4)

        # Highlight header row
        for (row_idx, col_idx), cell in table.get_celld().items():
            if row_idx == 0:
                cell.set_facecolor('#d3d3d3')
                cell.get_text().set_weight('bold')

        ax2.set_title('Classification Report', fontsize=11, fontweight='bold', pad=10)

        plt.tight_layout(rect=[0, 0.03, 1, 0.93])
        
        # Append page to PDF
        pdf.savefig(fig)
        plt.close(fig)

print(f"Successfully generated multi-page evaluation PDF at: {pdf_output_path}")

Successfully generated multi-page evaluation PDF at: ../dataset/results/all_datasets_evaluation_report.pdf
